# Loss-Grid Computer: T4 Experiment Suite

Runs the full experiment chain on a Colab T4 GPU:
- **RQ1 Exp 1** – hybrid applicability sweep (crossover threshold detection)
- **RQ1 Exp 2A** – calibration selection at the midpoint slowdown
- **RQ1 Exp 2B** – calibration cache amortization over a 4-variant session
- **RQ2** – cross-task generalization (California Housing regression)
- **Task B** – row-GRU workload-style generalization showcase

**Runtime estimate on T4:**
- `sample_repeats=3`, `measure_without_cache=False`: ~4–6 hours
- `sample_repeats=3`, `measure_without_cache=True`: ~7–10 hours (adds 4× calibration runs)
- `sample_repeats=1`, `measure_without_cache=False`: ~2–3 hours (faster, less reliable estimates)

Results are saved to Google Drive as JSON.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Root folder in your Drive where assets live and results will be saved.
# Expected layout:
#   DRIVE_ROOT/
#     assets/
#       cifar-10-batches-py/    <- CIFAR-10 dataset
#       cifar10-resnet20-0.pkl
#       cifar10-row-gru-0.pkl    <- optional; auto-trained if missing
#       cifar10-resnet20-123.pkl
#       cifar10-resnet20-2023.pkl
#       cifar10-resnet20-123456.pkl
#       california-mlp-0.pkl    <- optional; auto-trained if missing
#       sklearn/                <- California Housing cache (auto-downloaded)
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'

## 2. Clone repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/loss-grid-computer'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hotz99/loss-grid-computer.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
!pip install -q torch torchvision scikit-learn pandas numpy matplotlib
print('Dependencies installed.')

## 3. Apply local patches

Creates any files that may be missing if the repo is behind the local working state.

In [ ]:
import os

# system_schema.py is a compatibility shim required by the import chain.
# src/__init__.py and several backends import from it; without it the
# entire src package fails to load.
shim_path = 'src/system_schema.py'
if not os.path.exists(shim_path):
    with open(shim_path, 'w') as f:
        f.write(
            'from src.schemas import (  # noqa: F401\n'
            '    DatasetSpec, GridSpec, HybridMode, MLTaskSpec, RunMode,\n'
            '    RunRequest, SchedulerRequest, SchedulerResult, SurfacePoint, VanillaMode,\n'
            ')\n'
        )
    print(f'Created {shim_path}')
else:
    print(f'{shim_path} already present')

## 4. Link assets from Drive

In [ ]:
import os

assets_src = f'{DRIVE_ROOT}/assets'
assets_dst = 'assets'

if not os.path.exists(assets_dst):
    if os.path.exists(assets_src):
        os.symlink(assets_src, assets_dst)
        print(f'Assets linked from Drive: {os.listdir(assets_dst)}')
    else:
        os.makedirs(assets_dst, exist_ok=True)
        print(f'WARNING: {assets_src} not found in Drive. Create this folder and add checkpoints.')
else:
    print(f'Assets already present: {os.listdir(assets_dst)}')

required = [
    'assets/cifar-10-batches-py',
    'assets/cifar10-resnet20-0.pkl',
    'assets/cifar10-resnet20-123.pkl',
    'assets/cifar10-resnet20-2023.pkl',
    'assets/cifar10-resnet20-123456.pkl',
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print(f'MISSING required base assets (experiments will fail): {missing}')
else:
    print('All required assets present.')

## 5. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU found. Change Runtime > Change runtime type > T4 GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
import os
print(f'CPU cores: {os.cpu_count()}')

## 6. Train row-GRU with resumable Drive checkpoints

In [ ]:
import os
from pathlib import Path

import torch
from torch import nn

import scripts.train_row_gru as trg
from src.models.row_gru import RowGRUClassifier

row_gru_checkpoint = Path('assets/cifar10-row-gru-0.pkl')
drive_ckpt_dir = Path(f'{DRIVE_ROOT}/checkpoints/row-gru')
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

resume_state_path = drive_ckpt_dir / 'training_state.pt'
best_state_path = drive_ckpt_dir / 'best_state.pt'
drive_final_checkpoint = Path(f'{DRIVE_ROOT}/assets/cifar10-row-gru-0.pkl')
drive_final_checkpoint.parent.mkdir(parents=True, exist_ok=True)

if row_gru_checkpoint.exists():
    print(f'{row_gru_checkpoint} already present')
else:
    print(f'{row_gru_checkpoint} missing; training row-GRU on the T4 GPU with epoch checkpoints on Drive.')

    # Hyperparameters match scripts/train_row_gru.py defaults.
    epochs = 80
    batch_size = 512
    lr = 3e-3
    weight_decay = 1e-2
    label_smoothing = 0.05
    val_size = 5000
    num_workers = 2
    seed = 0
    max_train_samples = 0
    amp = True

    trg.set_seed(seed)
    device = trg.resolve_device('cuda')
    train_dataset, val_dataset, test_dataset = trg.build_datasets(Path('assets'), val_size, seed, max_train_samples)
    train_loader = trg.build_loader(train_dataset, batch_size, True, device, num_workers)
    val_loader = trg.build_loader(val_dataset, batch_size, False, device, num_workers)
    test_loader = trg.build_loader(test_dataset, batch_size, False, device, num_workers)

    model = RowGRUClassifier().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and amp)

    start_epoch = 1
    best_val_accuracy = -1.0
    best_state = None

    if resume_state_path.exists():
        checkpoint = torch.load(resume_state_path, map_location='cpu')
        trg.unwrap_model(model).load_state_dict(checkpoint['model_state'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        best_val_accuracy = float(checkpoint['best_val_accuracy'])
        best_state = checkpoint.get('best_state')
        print(f'Resumed from {resume_state_path} at epoch {start_epoch}.')

    for epoch in range(start_epoch, epochs + 1):
        train_loss, train_accuracy = trg.train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            device,
            amp,
        )
        scheduler.step()
        val_loss, val_accuracy = trg.evaluate(model, val_loader, criterion, device)
        print(
            f'epoch={epoch:03d} '
            f'train_loss={train_loss:.4f} train_acc={train_accuracy:.4f} '
            f'val_loss={val_loss:.4f} val_acc={val_accuracy:.4f} '
            f'lr={scheduler.get_last_lr()[0]:.6g}',
            flush=True,
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = {
                key: value.detach().cpu()
                for key, value in trg.unwrap_model(model).state_dict().items()
            }
            torch.save(best_state, best_state_path)

        epoch_state = {
            'epoch': epoch,
            'model_state': trg.unwrap_model(model).state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(),
            'best_val_accuracy': best_val_accuracy,
            'best_state': best_state,
        }
        tmp_resume = resume_state_path.with_suffix('.tmp')
        torch.save(epoch_state, tmp_resume)
        os.replace(tmp_resume, resume_state_path)

    if best_state is None and best_state_path.exists():
        best_state = torch.load(best_state_path, map_location='cpu')
    if best_state is None:
        raise RuntimeError('Training produced no checkpoint state')

    trg.unwrap_model(model).load_state_dict(best_state, strict=True)
    test_loss, test_accuracy = trg.evaluate(model, test_loader, criterion, device)
    torch.save(best_state, row_gru_checkpoint)
    torch.save(best_state, drive_final_checkpoint)
    print(
        f'saved={row_gru_checkpoint} and {drive_final_checkpoint} '
        f'best_val_acc={best_val_accuracy:.4f} '
        f'test_loss={test_loss:.4f} test_acc={test_accuracy:.4f}',
        flush=True,
    )

## 7. Experiment configuration

Edit this cell before running. The defaults are appropriate for a T4 Colab session.

In [ ]:
# --- Reproducibility ---
SEED = 1337

# --- Grid and batch ---
GPU_BATCH_SIZE = 64
GRID_RESOLUTION = 8       # 8x8 = 64 points per landscape

# --- RQ1 applicability sweep ---
MAX_SLOWDOWN = 100.0      # upper bound for GPU slowdown search
JUMP_FACTOR = 1.8         # multiplicative step during bracketing
LINEAR_SAMPLES = 5        # points sampled inside crossover interval

# bracket_repeats: runs per slowdown point during adaptive bracketing (keep 1 for speed)
# sample_repeats:  runs per slowdown point during final linear sampling (>=3 recommended)
BRACKET_REPEATS = 1
SAMPLE_REPEATS = 3

# --- Surface validation ---
ATOL = 1e-6
RTOL = 1e-5

# --- Experiment 2B ---
# True: run empirical without-cache session (accurate but adds ~4x calibration time)
# False: derive without-cache cost analytically from single-variant measurement
MEASURE_WITHOUT_CACHE = True

print('Configuration ready.')

## 8. Run experiments

This cell runs the full pipeline. Output is streamed to stdout and saved to Drive on completion.

In [ ]:
import sys, time

if '' not in sys.path:
    sys.path.insert(0, '')

# Invalidate any cached module state from a previous run in this session.
for mod in list(sys.modules.keys()):
    if mod.startswith('src') or mod.startswith('scripts'):
        del sys.modules[mod]

from scripts.experiment_pipeline import pipeline

start = time.perf_counter()
results = pipeline(
    seed=SEED,
    gpu_batch_size=GPU_BATCH_SIZE,
    grid_resolution=GRID_RESOLUTION,
    bracket_repeats=BRACKET_REPEATS,
    sample_repeats=SAMPLE_REPEATS,
    max_slowdown=MAX_SLOWDOWN,
    jump_factor=JUMP_FACTOR,
    linear_samples=LINEAR_SAMPLES,
    atol=ATOL,
    rtol=RTOL,
    measure_without_cache=MEASURE_WITHOUT_CACHE,
)
elapsed = time.perf_counter() - start
print(f'\nTotal wall time: {elapsed / 3600:.2f} h')

## 9. Save results to Drive

In [ ]:
import os
from src.results import to_pretty_json

os.makedirs(DRIVE_ROOT, exist_ok=True)
output_path = f'{DRIVE_ROOT}/t4-results.json'

with open(output_path, 'w') as f:
    f.write(to_pretty_json(results))

print(f'Results saved to {output_path}')

## 10. Key metrics summary

In [ ]:
showcases = results['workload_showcases']

for workload, data in showcases.items():
    rq1 = data['rq1']
    regime = data['fixed_regime']
    print(f'\n=== {workload} ===')
    print(f'  crossover region:  {rq1["crossover_region"]}')
    print(f'  midpoint slowdown: {regime["midpoint_slowdown"]:.3f}')
    print(f'  vanilla time:      {regime["vanilla_total_s"]:.1f}s')

    cal = regime['calibrated_hybrid']
    sta = regime['static_hybrid_baseline']
    print(f'  calibrated hybrid: {cal["total_s"]:.1f}s  config={cal["selected_mode"]}'
          f'  speedup={cal["surface_equivalence"]["speedup_vs_vanilla"]:.3f}'
          f'  valid={cal["surface_equivalence"]["allclose"]}')
    print(f'  static baseline:   {sta["total_s"]:.1f}s  config={sta["config"]}'
          f'  speedup={sta["surface_equivalence"]["speedup_vs_vanilla"]:.3f}'
          f'  valid={sta["surface_equivalence"]["allclose"]}')

    print('  samples:')
    for s in rq1['samples']:
        std = f'  std={s["speedup_std"]:.3f}' if s.get('speedup_std') is not None else ''
        print(f'    slowdown={s["slowdown"]:.2f}  speedup={s["speedup_mean"]:.3f}{std}'
              f'  wins={s["hybrid_wins"]}  valid={s["surface_valid"]}')

print('\n=== Experiment 2B: Cache Amortization ===')
exp2b = results['experiment_2b']
w = exp2b['with_cache_session']
wo = exp2b['without_cache_session']
print(f'  slowdown:             {exp2b["setup"]["gpu_slowdown_factor"]:.3f}')
print(f'  with cache:           {w["session_total_s"]:.1f}s'
      f'  (baseline={w["baseline_s"]:.1f}s  cal={w["calibration_s"]:.1f}s  exec={w["execution_s"]:.1f}s)')
print(f'  without cache [{wo["source"]}]:  {wo["session_total_s"]:.1f}s'
      f'  (baseline={wo["baseline_s"]:.1f}s  cal={wo["calibration_s"]:.1f}s  exec={wo["execution_s"]:.1f}s)')
saving_pct = 100 * (1 - w['session_total_s'] / wo['session_total_s'])
print(f'  session saving:       {saving_pct:.1f}%')